In [1]:
%%capture output
! pip install ibllib
! pip install pynapple
! pip install git+https://github.com/int-brain-lab/paper-brain-wide-map.git
! pip install -U google-colab

In [7]:
# system
from pathlib import Path
from tqdm import tqdm
import pickle

# analysis
import numpy as np
import pandas as pd
import pynapple as nap
from scipy import stats
import glob 
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
import os

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm


# set the IBL style for figures
# from ibl_style.style import figure_style
# figure_style()
from reproducible_ephys_functions import get_insertions
from functions import idxs_from_files
from one.api import ONE
one= ONE()

In [8]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'


# Load trial data

In [5]:
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
# session_cluster = pd.read_parquet(data_path+'cluster_per_session')
trial_modes = pd.read_parquet(data_path+'18_trial_waterclust')
trial_modes = trial_modes[['session', 'sample', 'trial_cluster']].drop_duplicates()
trial_modes['index'] = trial_modes['sample'].str.split().str[1:2].str.join('').astype(float)
trial_modes = trial_modes.rename(columns={'index':'trial_id', 'session':'eid'})

In [9]:
insertions = get_insertions(level=0, one=one, freeze='freeze_2024_03')
repro_ephys_insertions = pd.DataFrame.from_dict(insertions)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/2_pre-trial/data_releases/freeze_2024_03.csv'

In [ ]:
repro_eid = repro_ephys_insertions['probe_insertion']

In [ ]:
import brainwidemap
# this dataframe holds a summary of all the sessions
# and for us importantly, the eids and pids
bwm_df = brainwidemap.bwm_query()  # each row of this dataframe is a recording

n_sessions = bwm_df["eid"].unique().shape[0]
n_insertions = bwm_df["pid"].unique().shape[0]
print(
    f"{n_sessions} sessions with {n_insertions} individual neuropixel recordings"
)
# bwm_df.head()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


In [ ]:
bwm_pid = bwm_df['pid'].unique()

## Getting units from the brainwide map

In [ ]:
units_df = brainwidemap.bwm_units(one)
n_units = units_df.shape[0]
print(f"{n_units} units present in the table")
# units_df.head()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
62990 units present in the table


In [ ]:
# use_eid = lda_pid
use_eid = repro_eid
# use_eid = bwm_pid

In [ ]:
repro_units = units_df.loc[units_df['pid'].isin(list(use_eid))]

In [ ]:
# Here we repeat the imports and instantiate ONE, so this cell can also be run stand-alone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.ndimage

from iblutil.numerical import bincount2D
from iblatlas.atlas import BrainRegions
from one.api import ONE
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from ibl_style.style import figure_style

figure_style()  # set the style for IBL figures

one_kwargs = dict(
    base_url='https://openalyx.internationalbrainlab.org',
    username='intbrainlab',
    password='international',
    silent=True,
)
one = ONE(**one_kwargs)

def psth_indices(times, t_events, fs=None, event_window=np.array([-1, 1])):
    """
    Compute the peri-event time indices
    :param signal:
    :param times:
    :param t_events:
    :param fs:
    :param event_window:
    :return: idx_psth: (nc, nt), tscale
    """
    event_window = np.array(event_window)
    if fs is None:
        fs = 1 / np.nanmedian(np.diff(times))
    # compute a vector of indices corresponding to the perievent window at the given sampling rate
    sample_window = np.round(
        np.arange(event_window[0] * fs, event_window[1] * fs + 1)
    ).astype(int)
    # we inflate this vector to a 2d array where each column corresponds to an event
    idx_psth = np.tile(sample_window[:, np.newaxis], (1, t_events.size))
    # we add the index of each event too their respective column
    idx_event = np.searchsorted(times, t_events)
    idx_psth += idx_event
    # here we handle the case where the event window is outside of the initial boundaries
    i_out_of_bounds = np.logical_or(idx_psth > (times.size - 1), idx_psth < 0)
    idx_psth[i_out_of_bounds] = -1
    return idx_psth, sample_window.astype(float) / fs


def compute_binned_psths(st, sc, t_events, dt=0.02, event_window=np.array([-1, 1])):
    raster, t_scale, c_scale = bincount2D(st, sc, xbin=dt)
    ipsth, tscale = psth_indices(t_scale, t_events=t_events, event_window=event_window)
    return raster[:, ipsth], tscale  # (nc, nt, ne)

def extract_perievent(raster, t_scale, t_events, event_window):
    ipsth, tscale = psth_indices(
        t_scale, t_events=t_events, event_window=event_window
    )
    return raster[:, ipsth], tscale  # (nc, nt, ne)

def compute_binned_psths_with_baseline(
    st,
    sc,
    t_events,
    t_baseline_events,
    dt=0.02,
    event_window=np.array([-1, 1]),
    baseline_window=np.array([-0.5, 0])
):
    # bin spikes
    raster, t_scale, c_scale = bincount2D(st, sc, xbin=dt)

    # PSTH aligned to main event
    psth, tscale = extract_perievent(
        raster, t_scale, t_events, event_window)  # (nc, nt, ne)

    # baseline aligned to different event
    baseline, _ = extract_perievent(
        raster, t_scale, t_baseline_events, baseline_window)  # (nc, ntb, ne_b)

    # mean baseline per neuron
    # baseline_mean = np.nanmean(baseline, axis=(1, 2), keepdims=True)
    baseline_mean = np.nanmean(baseline, axis=1, keepdims=True)
    # shape: (nc, 1, 1)

    psth_bs = psth - baseline_mean

    return psth_bs, tscale, baseline_mean

## Querying and loading the data

Then we will query the brain region ACA. ACA has many sub-regions, and we need to get the list of neurons belinging to any of the sub-regions.

In [ ]:
BRAIN_REGIONS = ['PPC', 'CA1', 'DG', 'LP', 'PO']  # Reproducible ephys regions
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']  # Reproducible ephys regions

regions = BrainRegions()
aca_leaf_nodes = regions.descendants(regions.acronym2id(BRAIN_REGIONS))
# print(f"List of regions to query: \n {aca_leaf_nodes['acronym']}")
BRAIN_REGIONS = repro_units['Beryl'].unique()  # All available in the session query

List of regions to query: 
 ['VISam' 'VISam1' 'VISam2/3' 'VISam4' 'VISam5' 'VISam6a' 'VISam6b' 'VISa'
 'VISa1' 'VISa2/3' 'VISa4' 'VISa5' 'VISa6a' 'VISa6b' 'CA1' 'CA1slm'
 'CA1so' 'CA1sp' 'CA1sr' 'DG' 'DG-mo' 'DG-po' 'DG-sg' 'DG-sgz' 'DGcr'
 'DGcr-mo' 'DGcr-po' 'DGcr-sg' 'DGlb' 'DGlb-mo' 'DGlb-po' 'DGlb-sg' 'DGmb'
 'DGmb-mo' 'DGmb-po' 'DGmb-sg' 'LP' 'PO']


# Process and save PETH with trial info

In [4]:
# LOAD DATA
data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

save_states_path = prefix + 'representation_learning_variability/paper-individuality/data/states_files/'

NameError: name 'os' is not defined

# Individual sessions

In [15]:
# Identify sessions available to process
sessions_to_process = []
for m, mat in enumerate(idxs):
    mouse_name = mat[37:]
    session = mat[:36]
    sessions_to_process.append((mouse_name, session))
print(f"Found {len(sessions_to_process)} sessions to process.")

Found 319 sessions to process.


In [38]:
def calculate_cv_r2_random(data, formula, groups_col=None, n_folds=5, random_state=42):
    """Calculate cross-validated R² for model using random splits"""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    r2_scores = []
    
    # Convert data to array indices for splitting
    data_indices = np.arange(len(data))
    
    for train_idx, test_idx in kf.split(data_indices):
        train_data = data.iloc[train_idx]
        test_data = data.iloc[test_idx]
        
        if len(train_data) > 10 and len(test_data) > 5:  # Minimum sample sizes
            try:
                if groups_col is not None:
                    # Use mixed effects model with groups
                    model = mixedlm(formula, train_data, groups=train_data[groups_col])
                    result = model.fit(method='lbfgs')
                else:
                    # Use regular linear regression without groups
                    model = smf.ols(formula, train_data)
                    result = model.fit()
                
                # Predict on test data
                y_true = test_data[formula.split('~')[0].strip()]
                y_pred = result.predict(test_data)
                
                # Calculate R²
                r2 = r2_score(y_true, y_pred)
                r2_scores.append(r2)
                
            except Exception as e:
                print(f"Fold failed: {e}")
                continue
    
    return r2_scores


def run_models(states_trial_with_spikes, area, pid, session, cv_results):
    # Prepare data for all three models (same as before)
    data_categorical = states_trial_with_spikes.copy()
    data_categorical['paw'] = data_categorical['paw'].astype('category')
    data_categorical['whisk'] = data_categorical['whisk'].astype('category') 
    data_categorical['lick'] = data_categorical['lick'].astype('category')

    data_continuous = states_trial_with_spikes.copy()
    data_continuous['Lick count'] = data_continuous['Lick count'].astype('category')

    data_joint = states_trial_with_spikes.copy()
    data_joint['whisk'] = data_joint['whisk'].astype('category')
    data_joint['lick'] = data_joint['lick'].astype('category')

    spike_col = f'{area}_spike_count'
    
    # Filter data for current session
    session_data_cat = data_categorical[data_categorical['session_pid'] == pid]
    session_data_cont = data_continuous[data_continuous['session_pid'] == pid]
    session_data_joint = data_joint[data_joint['session_pid'] == pid]
    
    # Categorical model
    cat_data = session_data_cat.dropna(subset=['paw', 'whisk', 'lick', spike_col])
    if len(cat_data) > 50:  # Minimum data requirement
        formula_cat = f'{spike_col} ~ C(paw) + C(whisk) + C(lick)'
        # Now use random CV since we're within a single session
        r2_cat = calculate_cv_r2_random(cat_data, formula_cat, groups_col=None)
        
        if r2_cat:
            cv_results['brain_region'].append(area)
            cv_results['session'].append(session)
            cv_results['model_type'].append('Syllables')
            cv_results['mean_r2'].append(np.mean(r2_cat))
    
    # Continuous model
    cont_data = session_data_cont.dropna(subset=['l_paw_y_velocity', 'r_paw_x_velocity', 
                                                'r_paw_y_velocity', 'r_paw_x_velocity', 
                                                'whisker_me', 'Lick count', spike_col])
    if len(cont_data) > 50:
        formula_cont = f'{spike_col} ~ l_paw_y_velocity + r_paw_x_velocity + r_paw_y_velocity + r_paw_x_velocity + whisker_me + C(Q("Lick count"))'
        r2_cont = calculate_cv_r2_random(cont_data, formula_cont, groups_col=None)
        
        if r2_cont:
            cv_results['brain_region'].append(area)
            cv_results['session'].append(session)
            cv_results['model_type'].append('Raw data')
            cv_results['mean_r2'].append(np.mean(r2_cont))
    
    # Joint model
    joint_data = session_data_joint.dropna(subset=['l_paw_y_velocity', 'r_paw_x_velocity', 
                                                    'r_paw_y_velocity', 'r_paw_x_velocity', 
                                                    'whisker_me', 'whisk', 'lick', spike_col])
    if len(joint_data) > 50:
        formula_joint = f'{spike_col} ~ l_paw_y_velocity + r_paw_x_velocity + r_paw_y_velocity + r_paw_x_velocity + whisker_me + paw + whisk + lick'
        r2_joint = calculate_cv_r2_random(joint_data, formula_joint, groups_col=None)
        
        if r2_joint:
            cv_results['brain_region'].append(area)
            cv_results['session'].append(session)
            cv_results['model_type'].append('Joint')
            cv_results['mean_r2'].append(np.mean(r2_joint))

    return cv_results


In [53]:
BRAIN_REGIONS = ['VISa', 'VISam', 'CA1', 'DG', 'LP', 'PO']  # Reproducible ephys regions

# Storage for results - now includes session
cv_results = {
    'brain_region': [],
    'session': [],
    'model_type': [],
    'mean_r2': []
}

for p, pid in enumerate(list(use_eid)):
    try:
        ssl = SpikeSortingLoader(one=one, pid=pid)
        eid = ssl.eid
        session_files = glob.glob(os.path.join(save_states_path, f"*{eid}*"))
        if session_files:
            session_file_path = session_files[0]  # Take the first (and only) match
        
        states_trial = pd.read_parquet(session_file_path)
        identifiable_states = states_trial["identifiable_states"] #.dropna()
        digits = identifiable_states.str.extract(r'(\d)(\d)(\d)').astype('Int64')
        digits.columns = ["paw", "whisk", "lick"]
        states_trial['paw'] = digits['paw']
        states_trial['whisk'] = digits['whisk']
        states_trial['lick'] = digits['lick']
        
        # Initialize the final dataframe with states_trial
        states_trial_with_spikes = states_trial.copy()
        # Compute velocities for paw position variables
        states_trial_with_spikes['l_paw_y_velocity'] = np.gradient(states_trial_with_spikes['l_paw_y'])
        states_trial_with_spikes['r_paw_x_velocity'] = np.gradient(states_trial_with_spikes['r_paw_x'])
        states_trial_with_spikes['r_paw_y_velocity'] = np.gradient(states_trial_with_spikes['r_paw_y'])
        states_trial_with_spikes['r_paw_x_velocity'] = np.gradient(states_trial_with_spikes['r_paw_x'])

        # Add session identifier
        states_trial_with_spikes['session_eid'] = eid
        states_trial_with_spikes['session_pid'] = pid
        
        # Assuming states_trial has 'Bin' column with timestamps and some bin width
        bin_width = states_trial['Bin'].diff().median()  # or specify manually
        
        for a, area in enumerate(BRAIN_REGIONS):  
            regions = BrainRegions()
            aca_leaf_nodes = regions.descendants(regions.acronym2id(area))
            spikes, clusters, channels = ssl.load_spike_sorting(good_units=True, revision='2025-05-26')
            df_clus = pd.DataFrame(ssl.merge_clusters(spikes, clusters, channels))
            
            # This is where we select the units belonging to any of the leaf nodes brain regions
            selection_clusters = df_clus['atlas_id'].isin(aca_leaf_nodes['id'])
            iclusters = np.where(selection_clusters)[0]
            # We extend the selection to the spikes that belong to the selected clusters
            ispikes = np.isin(spikes['clusters'], iclusters)
            st = spikes['times'][ispikes]
            sc = spikes['clusters'][ispikes]
            
            # Get the time bins from states_trial
            time_bins = states_trial['Bin'].values
            # Create bin edges for histogram
            # We need n+1 edges for n bins, so we need to extrapolate the bin edges
            bin_width = np.median(np.diff(time_bins))  # Estimate bin width (~1/60 = 0.0167 seconds)
            # Create bin edges: start from half bin width before first bin, end half bin width after last bin
            bin_edges = np.concatenate([
                [time_bins[0] - bin_width/2],  # Start edge
                time_bins[:-1] + bin_width/2,  # Middle edges
                [time_bins[-1] + bin_width/2]  # End edge
            ])
            # Filter spikes to only include those within the states_trial time range
            time_mask = (st >= bin_edges[0]) & (st <= bin_edges[-1])
            st_filtered = st[time_mask]
            sc_filtered = sc[time_mask]
            
            # Create spike counts for each cluster in each time bin
            spike_counts = []
            if len(np.unique(sc_filtered)) >= 10:
                # print(area, len(np.unique(sc_filtered)))
                for cluster_id in np.unique(sc_filtered):
                    cluster_spikes = st_filtered[sc_filtered == cluster_id]
                    # Use histogram to bin spikes into the same time bins as states_trial
                    counts, _ = np.histogram(cluster_spikes, bins=bin_edges)
                    spike_counts.append(counts)

                # Convert to array and take average across all clusters
                if len(spike_counts) > 0:
                    spike_counts_array = np.array(spike_counts)
                    avg_spike_counts = np.mean(spike_counts_array, axis=0)
                else:
                    # Handle case where no clusters are found for this brain region
                    avg_spike_counts = np.zeros(len(states_trial))

                # Add this brain region's spike counts as a new column
                states_trial_with_spikes[f'{area}_spike_count'] = avg_spike_counts
                
                """ RUN MODELS """
                cv_results = run_models(states_trial_with_spikes, area, pid, eid, cv_results)
    
    #     # Add this session's data to the list
    #     all_sessions_data.append(states_trial_with_spikes)
        print(f"Processed session {p+1}/{len(use_eid)}: {eid}")
    except:
        print(f"Failed to process session {p+1}/{len(use_eid)}: {eid}")

# # Stack all sessions into a single dataframe
# final_stacked_data = pd.concat(all_sessions_data, ignore_index=True)
# print(f"Final stacked dataframe shape: {final_stacked_data.shape}")

VISa 22
LP 59
PO 76
Processed session 1/89: a4a74102-2af5-45dc-9e41-ef7f5aed88be
Processed session 2/89: d57df551-6dcb-4242-9c72-b806cff5613a
VISa 29
DG 10
LP 69
PO 42
Processed session 3/89: 56b57c38-2699-4091-90a8-aba35103155e
Processed session 4/89: 6ab9d98c-b1e9-4574-b8fe-b9eec88097e0
Processed session 5/89: bb099402-fb31-4cfd-824e-1c97530a0875
Processed session 6/89: ca4ecb4c-4b60-4723-9b9e-2c54a6290a53
LP 43
PO 76
Processed session 7/89: 72cb5550-43b4-4ef0-add5-e4adfdfb5e02
CA1 10
LP 16
Processed session 8/89: 0c828385-6dd6-4842-a702-c5075f5f5e81
LP 72
PO 111
Processed session 9/89: 746d1902-fa59-4cab-b0aa-013be36060d5
LP 23
PO 21
Processed session 10/89: dac3a4c1-b666-4de0-87e8-8c514483cacf
CA1 18
Processed session 11/89: caa5dddc-9290-4e27-9f5e-575ba3598614
VISa 10
Processed session 12/89: a8a8af78-16de-4841-ab07-fde4b5281a03
Failed to process session 13/89: f115196e-8dfe-4d2a-8af3-8206d93c1729
VISa 26
DG 10
LP 47
Processed session 14/89: 0a018f12-ee06-4b11-97aa-bbbff5448e9f
LP